## Account Plan Creation


In [16]:
import enum
from pydantic import BaseModel
import requests
import json
import re
from typing import Dict, List, Optional, Tuple
from echo.settings import MAX_RETRIES
from echo.tools.web_scraping import extract_data_from_links
from echo import sqldb
from echo.indexing import add_data, IndexType


# Replace with your actual port if different from 3000
API_URL = "http://localhost:3000/api/search"

sqldb.create_table(
    f'''CREATE TABLE IF NOT EXISTS {IndexType.BUYER_FOUNDATIONAL_PLAN.value} (
        seller TEXT,
        buyer TEXT,
        query_type TEXT,
        query TEXT,
        message TEXT,
        sources TEXT,
        source_extracted_data TEXT DEFAULT NULL,
        timestamp TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
        PRIMARY KEY (buyer, query_type)
    );'''   
)


class Metadata(BaseModel):
    title: str
    url: str

class Source(BaseModel):
    pageContent: str
    metadata: Metadata


class ExtractedData(BaseModel):
    link: str
    data: str


class CitedSource(BaseModel):
    citation_id: int
    title: str
    content: str
    url: str
    

class ExtractedCitedSource(CitedSource):
    data: Optional[str] = None
    

class SearchResponse(BaseModel):
    message: str
    sources: List[Source]
    buyer: str
    seller: str
    query_type: str
    query: str
    source_extracted_data: Optional[List[ExtractedCitedSource]] = None
    timestamp: Optional[str] = None


class SearchResult(BaseModel):
    message: str
    cited_content: List[str]
    


class QueryTypes(enum.Enum):
    FMOD = "Financial Moddeling"
    STRATEGY = "Strategic Initiatives"
    COMPANALYSIS = "Competitor Analysis"
    RECENTNEWS = "Recent News & Events"


query_type_prompts = {
    QueryTypes.FMOD.value: {
        "description": (
            "Financial Modelling: This information captures the financial information about the company. "
            "It includes the size of the company, industry, revenue, and other financial metrics. "
            "This information is crucial for understanding the company's financial health and potential as a client."
        ),
        "search": "Gather financial information including size, industry, and revenue about the company - {buyer}",
        "system": (
            "You are a financial analyst. "
            "You need to gather financial information about the company that is relevant to the user query. "
            "The information should be accurate and ONLY from the content provided. "
            "DO NOT make up any information."
        ),
        "user": (
            "Below is the information about {buyer}:\n{company_info} "
            "You are provided with content of the webpage: {webpage}. "
            "\n---CONTENT---\n{content}\n"
            "\n---END CONTENT---\n"
            "Extract the financial information like revenue, growth plan, profit about the company."
        )
    },
    QueryTypes.STRATEGY.value: {
        "description": (
            "Strategic Initiatives: This information captures the strategic initiatives of the company. "
            "It includes the company's key business priorities, strategic initiatives, and future plans. "
            "This information is crucial for understanding the company's direction and potential as a client."
        ),
        "search": (
            "Identify the company's key business priorities and strategic initiatives for the company - {buyer}. "
            "Analyze 10-K reports and financial statements. "
            "Gain insights into the company's operations, products, services, and market position. "
            "Assess the company's revenue, profitability, and overall financial stability to gauge its potential as a client."
            "Understand the company's future plans and priorities. "
            "Identify challenges the company faces, enabling you to position your product or service as a solution to mitigate these risks. "
        ),
        "system": (
            "You are a strategic analyst. "
            "You need to gather strategic initiatives about the company that is relevant to the user query. "
            "Find the information from annual reports and financial statements of the company. "
            "The information should be accurate and ONLY from the content provided. "
            "DO NOT make up any information."
        ),
        "user": (
            "Below is the information about {buyer}:\n{company_info} "
            
            "You are provided with content of the webpage: {webpage}. "
            "\n---CONTENT---\n{content}\n"
            "\n---END CONTENT---\n"
            
            "Identify the company's key business priorities and strategic initiatives for the company - {buyer}. "
            "Analyze 10-K reports and financial statements. "
            "Gain insights into the company's operations, products, services, and market position. "
            "Assess the company's revenue, profitability, and overall financial stability to gauge its potential as a client."
            "Understand the company's future plans and priorities. "
            "Identify challenges the company faces, enabling you to position your product or service as a solution to mitigate these risks. "
        )
    },
    QueryTypes.RECENTNEWS.value: {
        "description": (
            "Recent News & Events: This information captures the recent news and events about the company. "
            "It includes any recent developments, announcements, or changes that may impact the company's operations or strategy. "
            "This information is crucial for understanding the company's current position and future outlook."
        ),
        "search": (
            "Gather recent news and events about the company - {buyer}. "
            "Identify any recent developments, announcements, or changes that may impact the company's operations or strategy. "
            "This information is crucial for understanding the company's current position and future outlook."
        ),
        "system": (
            "You are a news analyst. "
            "You need to gather recent news and events about the company that is relevant to the user query. "
            "The information should be accurate and ONLY from the content provided. "
            "DO NOT make up any information."
        ),
        "user": (
            "Below is the information about {buyer}:\n{company_info} "
            "You are provided with content of the webpage: {webpage}. "
            "\n---CONTENT---\n{content}\n"
            "\n---END CONTENT---\n"
            
            "Gather recent news and events about the company - {buyer}. "
            "Identify any recent developments, announcements, or changes that may impact the company's operations or strategy. "
            "This information is crucial for understanding the company's current position and future outlook."
        )    
    },
    QueryTypes.COMPANALYSIS.value: {
        "description": (
            "Competitor Analysis: This information captures the competitors of the company. "
            "It includes the key players in the industry and their market positions. "
            "This information is crucial for understanding the competitive landscape and positioning your product or service."
        ),
        "search": (
            "Gather information about the competitors of the company - {buyer}. "
            "Identify key players in the industry and their market positions. "
            "Use websites like G2 or Crunchbase to find competitors. "
            "Analyze their strengths, weaknesses, and strategies to understand the competitive landscape."
        ),
        "system": (
            "You are a competitive analyst. "
            "You need to gather information about the competitors of the company that is relevant to the user query. "
            "The information should be accurate and ONLY from the content provided. "
            "DO NOT make up any information."
        ),
        "user": (
            "Below is the information about {buyer}:\n{company_info} "
            "You are provided with content of the webpage: {webpage}. "
            "\n---CONTENT---\n{content}\n"
            "\n---END CONTENT---\n"
            
            "Gather information about the competitors of the company - {buyer}. "
            "Identify key players in the industry and their market positions. "
            "Analyze their strengths, weaknesses, and strategies to understand the competitive landscape."
        )
    }
}


def curate_buyer_index_data_from_search_response(search_response: SearchResponse) -> Tuple[str, Dict]:
    data = []
    metadata = {
        "seller": search_response.seller,
        "buyer": search_response.buyer,
        "query_type": search_response.query_type,
        "query": search_response.query,
        "sources": [{
            "title": source.metadata.title,
            "url": source.metadata.url
        } for source in search_response.sources]
    }
    data = search_response.message
    return data, metadata


def curate_buyer_index_sources_data_from_search_response(search_response: SearchResponse) -> Tuple[str, Dict]:
    data = "\n\n".join([source.pageContent for source in search_response.sources])
    metadata = {
        "seller": search_response.seller,
        "buyer": search_response.buyer,
        "query_type": search_response.query_type,
        "query": search_response.query,
        "sources": [{
            "title": source.metadata.title,
            "url": source.metadata.url
        } for source in search_response.sources]
    }
    return data, metadata


def search_query(seller, buyer, query_type, history=None) -> SearchResponse:
    
    condition_dict = {
        "seller": seller, 
        "buyer": buyer, 
        "query_type": query_type
    }
    print("Checking if record exists in the database...")
    if sqldb.check_record_exists(IndexType.BUYER_FOUNDATIONAL_PLAN.value, condition_dict):
        print("Record exists, fetching from the database...")
        return SearchResponse(**sqldb.get_record(IndexType.BUYER_FOUNDATIONAL_PLAN.value, condition_dict))
    
    print("Record does not exist, making API call...")
    
    query = query_type_prompts[query_type]['search'].format(buyer=buyer)
    
    if history is None:
        history = [
            ["human", "Hi, how are you?"],
            ["assistant", "I am doing well, how can I help you today?"]
        ]
    
    headers = {"Content-Type": "application/json"}
    payload = {
        "chatModel": {
            "provider": "openai",
            "name": "gpt-4o-mini"
        },
        "embeddingModel": {
            "provider": "openai",
            "name": "text-embedding-3-large"
        },
        "optimizationMode": "speed",
        "focusMode": "webSearch",
        "query": query,
        "history": history
    }
    
    response = requests.post(API_URL, headers=headers, data=json.dumps(payload)).json()
    response_obj = SearchResponse(**{
        **response,
        "buyer": buyer,
        "seller": seller,
        "query_type": query_type,
        "query": query
    })
    description = query_type_prompts[query_type]['description']
    response_obj.message = f"{description}\n{response['message']}"
    
    sqldb.insert_record(
        IndexType.BUYER_FOUNDATIONAL_PLAN.value,
        {
            "buyer": buyer,
            "seller": seller,
            "query_type": query_type,
            "query": query,
            "message": response_obj.message,
            "sources": response['sources'],
        }
    )
    
    data, metadata = curate_buyer_index_data_from_search_response(response_obj)
    
    add_data(
        data=data,
        metadata=metadata,
        index_name=seller,
        index_type=IndexType.BUYER_FOUNDATIONAL_PLAN,
    )
    
    return response_obj


def get_cited_sources(source):
    pattern = r'\[([^\]]+)\]'
    matches: List[str] = re.findall(pattern, source)
    return list(set([int(i) for i in matches if i.isnumeric()]))


def get_cited_content(sources: List[Source], citations: List[int]) -> List[CitedSource]:
    cited_content = []
    for citation in citations:
        if citation < 0 or citation >= len(sources):
            continue
        content = sources[citation].pageContent
        title = sources[citation].metadata.title
        url = sources[citation].metadata.url
        cited_content.append(CitedSource(
            citation_id=citation,
            title=title,
            content=content,
            url=url
        ))
    return cited_content


def extract_data_from_sources(search_response: SearchResponse) -> SearchResponse:
    condition_dict = {
        "seller": search_response.seller, 
        "buyer": search_response.buyer, 
        "query_type": search_response.query_type
    }
    print("Checking if record exists in the database...")
    if sqldb.check_record_exists(IndexType.BUYER_FOUNDATIONAL_PLAN.value, condition_dict):
        print("Record exists, fetching from the database...")
        record = sqldb.get_record(IndexType.BUYER_FOUNDATIONAL_PLAN.value, condition_dict)
        if record['source_extracted_data']:
            return SearchResponse(**record)
    
    def make_search_call():
        print("Record does not exist, making API call...")
        citations = get_cited_sources(search_response.message)
        if not citations:
            data = []
        else:
            cited_sources = get_cited_content(search_response.sources, citations)
            print("cited sources", cited_sources)
            links = [source.url for source in cited_sources]
            company_info = search_response.message
            system_prompt = query_type_prompts[search_response.query_type]['system']
            user_prompt = query_type_prompts[search_response.query_type]['user'].format(
                buyer=search_response.buyer, company_info=company_info,
                webpage="{webpage}", content="{content}"
            )
            data = extract_data_from_links(links, user_prompt=user_prompt, system_prompt=system_prompt)
            extracted_data_map = {source['link']: source['data'] for source in data}
            extracted_cited_sources = [
                ExtractedCitedSource(
                    citation_id=citation.citation_id,
                    title=citation.title,
                    content=citation.content,
                    url=citation.url,
                    data=extracted_data_map[citation.url]
                )
                for citation in cited_sources
                if citation.url in extracted_data_map
            ]
        return extracted_cited_sources
    
    num_retries = MAX_RETRIES
    while num_retries > 0:
        try:
            extracted_cited_sources = make_search_call()
            break
        except requests.exceptions.RequestException as e:
            print(f"Request failed: {e}")
            num_retries -= 1
            if num_retries == 0:
                raise
        
    sqldb.update_record(
        IndexType.BUYER_FOUNDATIONAL_PLAN.value,
        condition_dict,
        {"source_extracted_data": [d.model_dump(mode='json') for d in extracted_cited_sources]}
    )
    
    sources_data, metadata = curate_buyer_index_sources_data_from_search_response(search_response)
    add_data(
        data=sources_data,
        metadata=metadata,
        index_name=search_response.seller,
        index_type=IndexType.BUYER_FOUNDATIONAL_PLAN,    
    )
    
    return search_response.model_copy(update={"source_extracted_data": extracted_cited_sources})


buyer = "Manpower group"
seller = 'Whatfix'

### Company Overview - landing page for buyer and seller in separate indexes. 

In [17]:
search_response = search_query(
    seller=seller,
    buyer=buyer, 
    query_type=QueryTypes.FMOD.value
)

Checking if record exists in the database...
Record exists, fetching from the database...


In [18]:
print(search_response.message)

Financial Modelling: This information captures the financial information about the company. It includes the size of the company, industry, revenue, and other financial metrics. This information is crucial for understanding the company's financial health and potential as a client.
ManpowerGroup Inc. is a prominent player in the workforce solutions and staffing industry, providing a wide range of services that cater to various employment needs. Below is a detailed overview of the company's financial information, including its size, industry position, and revenue.

## Company Overview

### Industry
ManpowerGroup operates within the staffing and workforce solutions industry, which encompasses recruitment, assessment, training, and outsourcing services. The company is recognized as one of the largest staffing firms globally, ranking third in the industry, following Adecco and Randstad[5].

### Size and Operations
ManpowerGroup has a significant global presence, with approximately 240 office

In [19]:
extracted_sources_response = extract_data_from_sources(search_response)

Checking if record exists in the database...
Record exists, fetching from the database...


### Strategic Initiatives - this should be part of landing page for buyer

In [20]:
search_response = search_query(seller=seller, buyer=buyer, query_type=QueryTypes.STRATEGY.value)

Checking if record exists in the database...
Record exists, fetching from the database...


In [21]:
print(search_response.message)

Strategic Initiatives: This information captures the strategic initiatives of the company. It includes the company's key business priorities, strategic initiatives, and future plans. This information is crucial for understanding the company's direction and potential as a client.
To analyze ManpowerGroup's key business priorities, strategic initiatives, and overall financial health, we can draw insights from their 10-K reports, financial statements, and other relevant documents. Below is a structured overview of the company's operations, market position, financial performance, and future plans, along with an assessment of the challenges it faces.

## Key Business Priorities

### 1. **Workforce Solutions**
ManpowerGroup focuses on providing comprehensive workforce solutions, including staffing, recruitment, assessment, and workforce consulting. Their services are designed to meet the evolving needs of clients across various industries, particularly in high-demand sectors such as IT, engi

In [23]:
extracted_sources_response = extract_data_from_sources(search_response)

Checking if record exists in the database...
Record exists, fetching from the database...
Record does not exist, making API call...
cited sources [CitedSource(citation_id=1, title='MANPOWERGROUP INC.', content='31 Dec 2024 — Outsourcing – We provide clients with outsourcing services related to human resources functions primarily in the areas of large-scale recruiting ...', url='https://investor.manpowergroup.com/static-files/b0756cfd-cb6f-4c0f-8a5a-742932e20a44'), CitedSource(citation_id=3, title='MANPOWERGROUP ENVIRONMENT, SOCIAL AND GOVERNANCE (ESG) REPORT 2020', content='in-demand roles in IT, engineering, logistics and digital manufacturing. We now have Manpower and Experis Academies in 10 key markets – including Australia, Belgium, Israel, Italy, Netherlands, Norway, Spain, Sweden, UK, and the U.S. – where we are helping thousands develop the skills our clients increasingly need.', url='https://www.manpowergroup.com/-/media/project/manpowergroup/mpg-marketing/pdf/sustainability/20

Summarizing Data From Links:   0%|          | 0/11 [00:00<?, ?it/s]

Failed to extract text from https://investor.manpowergroup.com/static-files/b0756cfd-cb6f-4c0f-8a5a-742932e20a44
Processing link: https://www.tradingview.com/news/tradingview:db119c720bc66:0-manpowergroup-inc-sec-10-q-report
Failed to extract text from https://www.manpowergroup.com/-/media/project/manpowergroup/mpg-marketing/pdf/sustainability/2020/mpg_esg_report_2020.pdf
Extracted data from https://www.manpowergroup.com
Extracted data from https://go.manpowergroup.com/year-in-review
Extracted data from https://www.marketresearch.com/MarketLine-v3883/ManpowerGroup-Strategy-SWOT-Corporate-Finance-33315362
Extracted data from https://www.manpowergroup.com/en/news-releases/news/manpowergroup-introduces-its-working-to-change-the-world-plan---reporting-esg-progress-and-ambitions-on-people--prosperity--planet-and-principles-of-governance
Extracted data from https://www.cipd.org/en/knowledge/factsheets/workforce-planning-factsheet
Extracted data from https://investor.manpowergroup.com/financi

### Recent News

In [24]:
search_response = search_query(seller=seller, buyer=buyer, query_type=QueryTypes.RECENTNEWS.value)
print(search_response.message)

Checking if record exists in the database...
Record does not exist, making API call...
Added node to index: Whatfix
Recent News & Events: This information captures the recent news and events about the company. It includes any recent developments, announcements, or changes that may impact the company's operations or strategy. This information is crucial for understanding the company's current position and future outlook.
ManpowerGroup, a leading workforce solutions company, has been active in recent months with several significant developments that could impact its operations and strategic direction. Here’s a summary of the latest news and events surrounding the company:

## Recent Financial Performance

### Fourth Quarter 2024 Results
On January 30, 2025, ManpowerGroup reported its fourth-quarter results, revealing a net earnings figure of $0.47 per diluted share for the three months ending December 31, 2024. This marked a notable recovery from a net loss of $1.73 per share in the same

In [25]:
extracted_sources_response = extract_data_from_sources(search_response)

Checking if record exists in the database...
Record exists, fetching from the database...
Record does not exist, making API call...
cited sources [CitedSource(citation_id=2, title='ManpowerGroup Reports 4th Quarter 2024 Results', content='30 Jan 2025 — ManpowerGroup (NYSE: MAN) today reported net earnings of $0.47 per diluted share for the three months ended December 31, 2024 compared to net losses of $1.73 ...', url='https://investor.manpowergroup.com/news-releases/news-release-details/manpowergroup-reports-4th-quarter-2024-results'), CitedSource(citation_id=3, title='ManpowerGroup Reports 3rd Quarter 2024 Results - PR Newswire', content='At the same time, our experienced management team continues to drive our key Diversification, Digitization and Innovation initiatives which are\xa0...', url='https://www.prnewswire.com/news-releases/manpowergroup-reports-3rd-quarter-2024-results-302278603.html'), CitedSource(citation_id=5, title='ManpowerGroup Reports 2nd Quarter 2024 Results - PR Ne

Summarizing Data From Links:   0%|          | 0/6 [00:00<?, ?it/s]

Failed to fetch page, status code: 403
Failed to extract text from https://www.investing.com/news/sec-filings/manpowergroup-announces-executive-reshuffle-93CH-3871533
Failed to fetch page, status code: 403
Failed to extract text from https://www.crunchbase.com/organization/manpowergroup/signals_and_news/timeline
Error processing link https://investor.manpowergroup.com/news-releases/news-release-details/manpowergroup-reports-4th-quarter-2024-results: Fireworks AI request failed: Server error '503 Service Unavailable' for url 'https://api.fireworks.ai/inference/v1/chat/completions'
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/503
Extracted data from https://www.prnewswire.com/news-releases/manpowergroup-reports-2nd-quarter-2024-results-302199951.html
Extracted data from https://en.wikipedia.org/wiki/ManpowerGroup
Extracted data from https://www.prnewswire.com/news-releases/manpowergroup-reports-3rd-quarter-2024-results-302278603.html
Added node to 

### Competitors

In [26]:
search_response = search_query(seller=seller, buyer=buyer, query_type=QueryTypes.COMPANALYSIS.value)
print(search_response.message)

Checking if record exists in the database...
Record does not exist, making API call...
Added node to index: Whatfix
Competitor Analysis: This information captures the competitors of the company. It includes the key players in the industry and their market positions. This information is crucial for understanding the competitive landscape and positioning your product or service.
To understand the competitive landscape surrounding ManpowerGroup, it is essential to identify its key competitors, analyze their market positions, and evaluate their strengths and weaknesses. Below is a detailed overview of the primary competitors in the staffing and workforce solutions industry, along with insights into their strategies.

## Key Competitors of ManpowerGroup

### 1. **Robert Half**
- **Market Position**: Robert Half is a leading staffing firm specializing in accounting, finance, IT, and administrative sectors. It is often viewed as a direct competitor to ManpowerGroup, particularly in the profes

In [27]:
extracted_sources_response = extract_data_from_sources(search_response)

Checking if record exists in the database...
Record exists, fetching from the database...
Record does not exist, making API call...
cited sources [CitedSource(citation_id=1, title='Manpowergroup Inc. - Company Profile Report - IBISWorld', content='Benchmark companies against industry averages, segment averages and their competitors. Identify real-world strengths, opportunities, weaknesses and threats for ...', url='https://www.ibisworld.com/united-states/company/manpowergroup-inc/422766'), CitedSource(citation_id=2, title='ManpowerGroup Competitors - Comparably', content='ManpowerGroup competitors include Volt Information Sciences, Inc., Randstad, TEKsystems, Kelly and Robert Half. ManpowerGroup ranks 1st in Diversity Score on Comparably vs its competitors. See below how ManpowerGroup compares to its competitors with CEO Rankings, Product & Services, NPS, Pricing, Customer Services, Overall Culture Score, eNPS ...', url='https://www.comparably.com/companies/manpowergroup/competitors'),

Summarizing Data From Links:   0%|          | 0/3 [00:00<?, ?it/s]

Failed to fetch page, status code: 403
Failed to extract text from https://www.comparably.com/companies/manpowergroup/competitors
Extracted data from https://www.globaldata.com/company-profile/manpowergroup-inc/competitors
Error processing link https://www.ibisworld.com/united-states/company/manpowergroup-inc/422766: An error occurred: The read operation timed out
Added node to index: Whatfix


In [28]:
from echo.queries import get_queries

queries = get_queries(seller=seller)